# M3-B1 — Exploration des 3 sources Acerox

> Notebook d'**exploration rapide** — pas d'EDA fouillée, juste assez pour
> remplir l'inventaire de la note d'identification.

Auteur·rice : `<prénom>` — Date : `<date>`

**Règles** :
- Pas de transformation (juste lecture, `info`, `head`, `describe`)
- Une cellule markdown par source — qu'est-ce que tu observes ?
- Trace les **risques** et **questions** qui émergent pour l'`identification_sources.md`

In [1]:
from pathlib import Path
import json

import pandas as pd

DATA_DIR = Path("../data")

## Source 1 — Capteurs IoT (CSV)

In [21]:
df_iot = pd.read_csv(DATA_DIR / "capteurs_iot.csv")
df_iot.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51000 entries, 0 to 50999
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   timestamp      51000 non-null  object 
 1   site           51000 non-null  object 
 2   line_id        51000 non-null  int64  
 3   sensor_id      51000 non-null  object 
 4   temperature_c  51000 non-null  float64
 5   vibration_mms  50251 non-null  float64
 6   debit_uh       51000 non-null  float64
dtypes: float64(3), int64(1), object(3)
memory usage: 2.7+ MB


In [58]:
# describe, head, value_counts par site/line, manquants
print(f"Volume : {len(df_iot):,} lignes × {len(df_iot.columns)} colonnes")
print(f"Période : {df_iot['timestamp'].min()} → {df_iot['timestamp'].max()}")

print("\describe :")
display(df_iot.describe())

print(f"\nValeurs manquantes pour les vibrations : {df_iot['vibration_mms'].isna().sum()}")
print("\nValeurs manquantes pour les vibrations par site :")
display(df_iot.groupby('site')['vibration_mms'].apply(lambda s: s.isna().sum()))

print("\nRépartition des données par site :")
display(df_iot.value_counts("site"))

print("Nombre de doublons (site,timestamp) :", df_iot.duplicated(subset=["site", "timestamp"], keep=False).sum())
print("Nombre de doublons (lignes,vibrations) :", df_iot.duplicated(subset=["line_id", "vibration_mms"], keep=False).sum())

print("\nVariabilité (écart-type) comparée entre sites :")
std_site = df_iot.groupby("site")[["vibration_mms", "temperature_c", "debit_uh"]].std().round(3)
display(std_site)

print("\nExemples de lignes :")
display(df_iot.head())

Volume : 51,000 lignes × 7 colonnes
Période : 2026-04-01T00:00:38 → 2026-04-29T23:57:53
\describe :


,line_id,temperature_c,vibration_mms,debit_uh
count,51000.000000,51000.000000,50251.000000,51000.000000
mean,2.005882,73.717034,4.831502,110.050188
std,1.033472,27.005577,2.685963,11.832749
min,1.000000,26.470000,0.000000,80.000000
25%,1.000000,60.237500,3.302777,102.000000
50%,2.000000,66.090000,4.187726,110.100000
75%,3.000000,72.840000,5.171534,118.050000
max,4.000000,160.000000,12.000000,150.000000



Valeurs manquantes pour les vibrations : 749

Valeurs manquantes pour les vibrations par site :


site
Lyon             160
Roubaix          289
Saint-Etienne    300
Name: vibration_mms, dtype: int64


Répartition des données par site :


site
Roubaix          20570
Saint-Etienne    20248
Lyon             10182
Name: count, dtype: int64

Nombre de doublons (site,timestamp) : 2379
Nombre de doublons (lignes,vibrations) : 7710

Variabilité (écart-type) comparée entre sites :


,vibration_mms,temperature_c,debit_uh
site,,,
Lyon,1.194,8.065,11.925
Roubaix,3.641,37.834,11.824
Saint-Etienne,1.198,8.050,11.794



Exemples de lignes :


,timestamp,site,line_id,sensor_id,temperature_c,vibration_mms,debit_uh
0,2026-04-14T19:21:43,Lyon,1,SLYO-L1-T01,77.92,5.539793,101.27
1,2026-04-27T02:47:12,Lyon,1,SLYO-L1-T01,70.58,3.361715,110.19
2,2026-04-13T18:18:50,Saint-Etienne,1,SSAI-L1-T01,62.37,4.019277,111.28
3,2026-04-05T10:34:03,Roubaix,2,SROU-L2-T01,66.17,4.922531,123.93
4,2026-04-20T10:18:07,Saint-Etienne,3,SSAI-L3-T01,55.56,1.643043,101.40


> **Observations** :
>
> - Volume : 51000 lignes
> - Période : environ 1 mois de données
> - Qualité observée : bonne qualité seul quelques manquements dans "vibration_mms" (à confirmer avec une analyse des faux manquants). Doublons par site sur des dates à creuser, de même pour les doublons sur une ligne pour les vibrations.
> - Risques RGPD : pour moi pas de risque
> - Pertinence métier : intéressant pour la prédiction notamment les données vibration_mms, temperature_c, debit_uh qui doivent être des paramètres qui permettent de détecter une potentielle future panne
> - Question pour Sébastien : Quels sont les critères déterminants pour définir un dafaut qualité ?

## Source 2 — ERP (JSON)

In [60]:
with (DATA_DIR / "erp_export.json").open() as f:
    orders = json.load(f)
df_erp = pd.DataFrame(orders)
print("\nInfo :")
display(df_erp.info())


Info :
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   ordre_id         2000 non-null   int64 
 1   produit_ref      2000 non-null   object
 2   site             2000 non-null   object
 3   line_id          2000 non-null   int64 
 4   date_lancement   2000 non-null   object
 5   date_fin_prevue  2000 non-null   object
 6   statut           2000 non-null   object
 7   ouvrier_id       1891 non-null   object
 8   quantite_kg      2000 non-null   int64 
dtypes: int64(3), object(6)
memory usage: 140.8+ KB


None

In [77]:
# explore : value_counts statut, manquants ouvrier_id, etc.
print(f"Volume : {len(df_erp):,} ordres")
print(f"Période : {df_erp['date_lancement'].min()} → {df_erp['date_lancement'].max()}")

print("\describe :")
display(df_iot.describe())

print(f"\nRépartition par statut :")
display(df_erp['statut'].value_counts())

# pour le statut "suspendu", tableau des produit_ref et de leur nombre d'occurrences
suspendus = df_erp[df_erp['statut'] == 'suspendu']['produit_ref']
print(f"\nRépartition des produits suspendus :")
display(suspendus.value_counts())
# pour le statut "annule", tableau des produit_ref et de leur nombre d'occurrences
annule = df_erp[df_erp['statut'] == 'annule']['produit_ref']
print(f"\nRépartition des produits annule :")
display(annule.value_counts())

print(f"\nValeurs manquantes pour 'ouvrier_id' : {df_erp['ouvrier_id'].isna().sum()}")  # ⚠️ RGPD-sensible

print("\nExemples de lignes :")
display(df_erp.head())

Volume : 2,000 ordres
Période : 2026-04-01T00:09:57 → 2026-04-29T23:20:33
\describe :


,line_id,temperature_c,vibration_mms,debit_uh
count,51000.000000,51000.000000,50251.000000,51000.000000
mean,2.005882,73.717034,4.831502,110.050188
std,1.033472,27.005577,2.685963,11.832749
min,1.000000,26.470000,0.000000,80.000000
25%,1.000000,60.237500,3.302777,102.000000
50%,2.000000,66.090000,4.187726,110.100000
75%,3.000000,72.840000,5.171534,118.050000
max,4.000000,160.000000,12.000000,150.000000



Répartition par statut :


statut
termine     1559
en_cours     197
suspendu     139
annule       105
Name: count, dtype: int64


Répartition des produits suspendus :


produit_ref
INOX-321-2    22
ALU-T2-25     17
INOX-304-2    16
ALU-T3-10     16
INOX-316-4    14
ALU-T1-22     13
ALU-T1-15     13
INOX-304-3    11
INOX-316-2     9
ALU-T2-18      8
Name: count, dtype: int64


Répartition des produits annule :


produit_ref
ALU-T2-25     14
INOX-316-2    14
ALU-T1-22     13
INOX-316-4    12
INOX-304-2    11
ALU-T1-15     10
ALU-T3-10     10
INOX-304-3     8
ALU-T2-18      7
INOX-321-2     6
Name: count, dtype: int64


Valeurs manquantes pour 'ouvrier_id' : 109

Exemples de lignes :


,ordre_id,produit_ref,site,line_id,date_lancement,date_fin_prevue,statut,ouvrier_id,quantite_kg
0,100000,ALU-T1-22,Roubaix,3,2026-04-01T22:21:08,2026-04-02T23:21:08,suspendu,EMP-5317,3221
1,100001,INOX-316-4,Saint-Etienne,1,2026-04-26T14:52:52,2026-04-28T15:52:52,termine,EMP-7240,4556
2,100002,ALU-T2-18,Saint-Etienne,3,2026-04-11T09:54:06,2026-04-12T16:54:06,suspendu,EMP-1939,1308
3,100003,ALU-T1-22,Roubaix,1,2026-04-20T22:33:08,2026-04-22T04:33:08,termine,EMP-3531,2968
4,100004,ALU-T2-25,Roubaix,4,2026-04-24T01:03:02,2026-04-25T21:03:02,termine,EMP-8778,3278


> **Observations** :
>
> - Volume : 2000 ordres
> - Période : environ 1 mois de données (même période que capteurs_iot.csv)
> - Qualité observée : structure globalement propre (ordre_id, line_id, quantite_kg bien renseignés), mais 109 valeurs manquantes sur ouvrier_id (~5,5%). Les statuts montrent une majorité de `termine` (1559), puis `en_cours` (197), `suspendu` (139) et `annule` (105).
> - ⚠️ Risque RGPD : `ouvrier_id` est une donnée indirectement identifiante (ex. EMP-5317), donc minimisation à prévoir.
> - Pertinence métier : intéressante pour relier la production aux résultats opérationnels (statut final, produit, ligne, site, quantités) et identifier les produits/lignes surreprésentés en `suspendu`/`annule`.
> - Question pour Sébastien : les statuts `suspendu` et `annule` sont-ils liés à des défauts qualité, à des causes logistiques, ou aux deux ?

## Source 3 — Logs machines (texte)

In [27]:
log_path = DATA_DIR / "logs_machines.log"
n_lines = sum(1 for _ in log_path.open())
print(f"Nombre de lignes : {n_lines:,}")
print(f"Taille fichier : {log_path.stat().st_size / 1024:.1f} Ko")

# Aperçu des 5 premières lignes
with log_path.open() as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        print(line.rstrip())

# On parcours le fichier pour chaque ligne la mettre dans un csv et remplacer les 4 premiers espaces par des virgules. On ne peut pas utiliser `sep=" "` pour lire le fichier car le message peut contenir des espaces.
# la première ligne du csv de sortie doit être `date`, `site`, `ligne_id`, `type d'événement`, `explication`
with log_path.open() as f:
    with (DATA_DIR / "logs_machines.csv").open("w") as f_out:
        f_out.write("date,site,ligne_id,type d'événement,explication\n")
        for line in f:
            # On remplace les 4 premiers espaces par des virgules
            line = line.replace(" ", ",", 4)
            f_out.write(line)

import re

log_encoding = "cp1252"
out_csv = DATA_DIR / "logs_machines.csv"

pattern = re.compile(
    r"^\[(?P<date>[^\]]+)\]\s+(?P<site>\S+)\s+LINE-(?P<line_id>\d+)\s+"
    r"(?P<event_type>INFO|WARN|ERROR):\s*(?P<explication>.*)$"
)

rows = []
with log_path.open(encoding=log_encoding, errors="replace") as f:
    for line in f:
        m = pattern.match(line.strip())
        if m:
            rows.append(m.groupdict())

df_machines = pd.DataFrame(rows)

Nombre de lignes : 30,000
Taille fichier : 1872.8 Ko
[2026-04-01T00:00:16] Lyon LINE-1 INFO: shift_changed
[2026-04-01T00:01:07] Saint-Etienne LINE-2 INFO: operator_login
[2026-04-01T00:01:34] Saint-Etienne LINE-3 ERROR: vibration_overlimit sensor=SSAI-L3-T01
[2026-04-01T00:04:18] Roubaix LINE-4 INFO: maintenance_completed
[2026-04-01T00:04:35] Lyon LINE-1 INFO: tooling_loaded


In [ ]:
# TODO — compter les types d'événements (INFO/WARN/ERROR), repérer les pics
df_machines.to_csv(out_csv, index=False, encoding="utf-8")

info = (df_machines["event_type"] == "INFO").sum()
warn = (df_machines["event_type"] == "WARN").sum()
error = (df_machines["event_type"] == "ERROR").sum()
print(f"INFO: {info:,}, WARN: {warn:,}, ERROR: {error:,}")

df_machines = pd.read_csv(out_csv, encoding="utf-8")

# describe, head, value_counts, manquants...
display(df_machines.describe())
display(df_machines.head())
display(df_machines.isna().sum())
display(df_machines.value_counts("event_type"))
display(df_machines.groupby("site")["event_type"].value_counts())

# Pics d'erreur sur roubaix ... TODO...
roubaix_errors = df_machines[(df_machines["site"] == "ROUBAIX") & (df_machines["event_type"] == "ERROR")]

INFO: 22,501, WARN: 5,758, ERROR: 1,741


,line_id
count,30000.000000
mean,1.999233
std,1.033538
min,1.000000
25%,1.000000
50%,2.000000
75%,3.000000
max,4.000000


,date,site,line_id,event_type,explication
0,2026-04-01T00:00:16,Lyon,1,INFO,shift_changed
1,2026-04-01T00:01:07,Saint-Etienne,2,INFO,operator_login
2,2026-04-01T00:01:34,Saint-Etienne,3,ERROR,vibration_overlimit sensor=SSAI-L3-T01
3,2026-04-01T00:04:18,Roubaix,4,INFO,maintenance_completed
4,2026-04-01T00:04:35,Lyon,1,INFO,tooling_loaded


date           0
site           0
line_id        0
event_type     0
explication    0
dtype: int64

event_type
INFO     22501
WARN      5758
ERROR     1741
Name: count, dtype: int64

site           event_type
Lyon           INFO          4661
               WARN          1072
               ERROR          256
Roubaix        INFO          8511
               WARN          2497
               ERROR          962
Saint-Etienne  INFO          9329
               WARN          2189
               ERROR          523
Name: count, dtype: int64

Nombre d'erreurs sur ROUBAIX : 0


> **Observations** :
>
> - Format : ...
> - Parsing nécessaire : ...
> - Croisement avec les capteurs IoT (corrélation Roubaix line 3 ?) : ...

## Synthèse pour `identification_sources.md`

Remplis le tableau d'inventaire dans `../identification_sources.md` à
partir des observations ci-dessus.